# TDWI Lab 3 Part 2: Running Cursor Cloud Agents

In this lesson you will run the broken sales pipeline locally, steer a **Cloud Agent** with **`AGENTS.md`**, review its **draft PR**, merge on GitHub, and confirm everything works on `main`.


## Learning Objectives

By the end of this mini-lesson you will be able to:
- Run the test suite and pipeline script to observe real failures
- Use `AGENTS.md` to guide a Cloud Agent's workflow
- Run a Cursor Cloud Agent to fix the pipeline
- Review a draft PR locally, merge on GitHub, and verify `main`
- Optionally, enable **GitHub MCP** and run a read-only summary of the draft PR on **your fork**


## Prerequisites

- Completed [README.md](README.md) setup (fork, clone, local `.venv`, test push)
- Completed [LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb](LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb) (Cloud environment, secrets on **your fork**)
- `.venv` activated in Cursor's terminal (prompt shows `(.venv)`)
- Working on **`main`** of **your fork** in Cursor

## Step 1: Run the test suite and observe failures

Open a terminal in the project root with `.venv` active, then run:

```bash
python -m pytest test_sales_report.py
```

You should see **failing tests** (for example: duplicate orders not removed, revenue not recalculated, dates not parsed). These tests describe what "fixed" means for this lab.

## Step 2: Run the pipeline script and observe issues

Run the main script:

```bash
python generate_sales_report.py
```

You may see additional problems beyond the tests, such as:
- **`REPORT_EXPORT_KEY` not set** locally (the script uses this environment variable at runtime)
- **`output/` directory missing** (export may fail until the code creates it)
- Incorrect metrics or chart output because the data is still messy

This is expected. The next step is to use a Cloud Agent to fix the pipeline in the environment you configured in Part 1.

## Step 3: Why a Cloud Agent — and why `AGENTS.md`?

In Part 1 you configured a **Dockerfile-managed Cloud Agent environment** so the agent runs in the same setup every time.

**`AGENTS.md`** is Cursor's repo-level instructions for agents. It tells the agent what this project is, how to run tests and the pipeline, and workflow rules you define.

Open [`AGENTS.md`](AGENTS.md) in Cursor and read the **Overview** and **Core Workflow Rules** sections.

### Add the testing workflow rule (together in class)

Under **Core Workflow Rules** → **Running tests**, after the `pytest` command block, add this paragraph:

```
Always run `python -m pytest test_sales_report.py` before pushing changes. Ensure all tests pass before pushing. After fixing code, re-run the tests to verify the fixes. If a test fails, diagnose the issue, fix it, and re-run the tests to verify. Iterate until all tests pass and you have met the user's requirements.
```

This tells the Cloud Agent to run tests, fix failures, and only push when green—without you repeating those instructions in every prompt.

This is **Recipe 2** in [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md): agents run deterministic tests—they don't replace them. That guide has more recipes (check scripts, slash commands, CI, automations) you can adopt after the lab.

**Harness vs what you wire:** Cursor's agent harness already orchestrates much of the inner loop—**planning**, exploring the repo, running terminal commands, iterating on failures. You saw those steps in Part 1 and will again in the Cloud Agent UI. `AGENTS.md` is your **team layer**: repo rules the harness won't infer on its own. See [Harness vs team workflow](WORKFLOW_RECIPES.md#harness-vs-team-workflow) in WORKFLOW_RECIPES.md.


## Step 4: Update `AGENTS.md`, then commit and push to your fork

If you or your instructor edited `AGENTS.md`, the Cloud Agent will only see those changes after they are on **GitHub**.

**You must commit and push `AGENTS.md` to your fork before starting the Cloud Agent in Step 5.**

1. Open the **Source Control** tab in Cursor.
2. Stage `AGENTS.md` (click **+** next to the file).
3. Enter a commit message, e.g. `Add AGENTS.md workflow rules for lab`.
4. Click **Commit**.
5. Click **Synchronize Changes** (or **Push**) to push to **your fork** on GitHub.

You should have added the testing workflow rule in Step 3. If you have not saved that change yet, add it now before committing.

## Step 5: Start a Cloud Agent with the fix prompt

1. Go to [cursor.com/agents](https://cursor.com/agents) (or the **Agents** window in Cursor).
2. Above the prompt, select **your fork** of this repository (not only the upstream template repo).
3. Paste this prompt and send:

```text
The generate_sales_report.py script is not working. It should create the png report and the csv. Also, all the tests are failing and the code needs to be fixed so that all tests pass. Do not modify README.md or any lab notebook.
```

## Step 6: Watch the agent run

The agent is writing and iterating — you may not be coding for several minutes. Use the wait to follow its progress, skim the emerging diff, and review what you already know about the failures. That is normal agentic work, not downtime.

1. Open the agent session and follow its progress (edits, terminal, pytest runs).
2. This may take several minutes. The agent uses your Part 1 Cloud environment.
3. Open the **Git**, **Terminal**, or **Desktop** tabs in the agent sidebar to see what it is doing.
4. When finished, the agent should open a **draft PR** on your fork.

### Optional: verify in the Cloud Agent environment

Before reviewing locally, you can confirm the fix in the agent's VM:

1. Open the **Terminal** tab in the agent session and run:

```bash
python -m pytest test_sales_report.py
python generate_sales_report.py
```

2. Open the **Desktop** tab. Click the **folder** icon at the bottom to open the workspace, then open `report.png`.

**Desktop tips (first time only):**
- You may be asked to create a **keyring password** — enter any value (e.g. `test`).
- Chrome may ask to be the **default browser** and the image may hang on first open. Accept the default-browser prompt, then open `report.png` again; it should load.


## Step 7: Open the draft PR on GitHub

1. Open your fork on GitHub.
2. Go to **Pull requests** and open the agent's PR (often a `cursor/...` branch into `main`).
3. Note that it is a **draft PR** — that is normal. You will review locally before merging.

## Optional: GitHub MCP

Do this after your draft PR exists (Step 7). You can skip it and continue to Step 8.

**MCP** (Model Context Protocol) = a standard way to plug **structured tools** into an agent host. GitHub MCP talks to the same GitHub you already use — issues, PRs, diffs — without inventing a new app.

Use GitHub’s **remote** hosted server (no Docker required). Official install guide: [Install GitHub MCP in Cursor](https://github.com/github/github-mcp-server/blob/main/docs/installation-guides/install-cursor.md).

### Enable (remote — recommended)

1. Reuse the **PAT** from Lab 3 setup (or create one with repo read access).
2. Open Cursor **Settings → Tools & MCP** (or edit `~/.cursor/mcp.json`).
3. Add a `github` server like this (replace the token; do **not** commit the PAT):

```json
{
  "mcpServers": {
    "github": {
      "url": "https://api.githubcopilot.com/mcp/",
      "headers": {
        "Authorization": "Bearer YOUR_GITHUB_PAT"
      }
    }
  }
}
```

4. Save, **fully restart Cursor**, then confirm a green status under MCP tools.
5. If remote setup fails, try the Docker local option in the official guide.

*Snapshot tip:* UI labels drift; if “Tools & MCP” moved, search Settings for MCP. Dated for July 2026.

### Read-only prompt (your fork)

With the **draft PR** open from Step 7, paste this in **local** Agent chat (your Cursor IDE — not the Cloud Agent session):

```text
Using the GitHub MCP, list open PRs on your fork and summarize what the Cloud Agent’s draft PR changes. Do not edit files or merge.
```

**One-line debrief:** MCP is a **connector pattern** — structured tools for the agent; same GitHub, less copy-paste.

**Then continue to Step 8** — MCP is a structured summary; local pytest and diff review are still required before merge.


## Optional: `gh` CLI vs GitHub MCP

Same goal, two tools — then compare. Skip this if you did not set up GitHub MCP.

Pick one task (for example: summarize open PRs on **your fork**). Run it twice:

1. *Using the `gh` CLI only (no MCP), …*
2. *Using the GitHub MCP only (do not call `gh`), …*

| | **`gh` CLI** | **GitHub MCP** |
|--|--------------|----------------|
| **What it is** | GitHub’s command-line tool; agent runs shell commands | MCP server exposing GitHub as structured tools the host wires in |
| **Pros** | Common in real workflows; easy to script/CI; transparent commands | Clean tool schema; less flag fuss; portable “any MCP host” idea |
| **Cons** | Agent must know flags; shell/auth quirks; not every machine has `gh` | Extra Cursor config; PAT/scopes; another moving part |
| **When you’d prefer it** | Automation, scripts, CI, terminal-first habits | IDE-centric agents; teaching MCP; tools without teaching CLI surface area |
| **Honest take** | Neither is universally better. Many teams standardize on `gh`. MCP is the **standard connector pattern** beyond one CLI. Learn both; pick per team. |


## Step 8: Pull the branch locally, run tests, and review the diff

In your local terminal (project root, `.venv` active):

```bash
git fetch origin
git branch -a
git checkout <agent-branch-name>
```

`git branch -a` lists local and remote branches — confirm you see `remotes/origin/<agent-branch-name>` before checking out.

Replace `<agent-branch-name>` with the branch name from the PR (shown on GitHub).

Run tests on the agent's branch:

```bash
python -m pytest test_sales_report.py
```

Review the code changes in Cursor (diff view or **Source Control**). Confirm the fixes look reasonable before merging.

## Step 9: Mark the PR ready and merge on GitHub

Draft PRs must be marked **Ready for review** before GitHub allows a merge.

1. On the PR page, click **Ready for review**.
2. Click **Merge pull request** and confirm the merge into `main` on **your fork**.
3. You can delete the agent branch on GitHub if prompted (optional).

## Step 10: Update `main` locally and verify everything works

After merging on GitHub, sync your local `main` and run the pipeline end-to-end:

```bash
git checkout main
git pull origin main
```

Set `REPORT_EXPORT_KEY` for this terminal session (any value is fine for the lab, e.g. `demo-123`):

**Mac / Linux**

```bash
export REPORT_EXPORT_KEY=demo-123
```

**Windows (PowerShell)**

```powershell
$env:REPORT_EXPORT_KEY = "demo-123"
```

Run tests and the script:

```bash
python -m pytest test_sales_report.py
python generate_sales_report.py
```

Tests should pass. The script should print metrics and create `report.png` and `output/encrypted_sales_report.csv`.

Part 2 is complete. Continue with [LAB3-Part-3-Automations.ipynb](LAB3-Part-3-Automations.ipynb).

## Debrief questions

1. What did `AGENTS.md` tell the agent that your one-line prompt did not?
2. What did you check locally before merging that the agent could not check for you?
3. Why push `AGENTS.md` to GitHub before starting the agent?
4. *(If you tried MCP)* Did GitHub MCP change how you looked at the draft PR — or did the browser already suffice?
5. *(If you compared)* For your team, would you standardize on **`gh`**, **MCP**, or both?
